# m40 lab · Segmentation metrics, synthetic atria and a real left-atrium data set

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You compute the metrics used to judge segmentations, check them against closed forms, segment synthetic atria with
a threshold and with a voxel classifier, measure what segmentation errors mean for points a catheter would touch,
and repeat the key measurements on 20 real cardiac MRI volumes with the left atrium outlined.

| Part | What you do | Data |
| --- | --- | --- |
| A | Metrics on shapes with known answers: eq-m40-sphere, eq-m40-slab, an island and the Hausdorff distance | synthetic |
| B | Synthetic atria: body, veins, appendage, a bright neighbour, bias and noise | synthetic |
| C | Threshold against a voxel classifier; Dice, HD95, ASSD and NSD; body against veins | synthetic |
| D | From mask to map: how far is the predicted shell from points on the true wall? | synthetic |
| E | Medical Segmentation Decathlon, left atrium: shape, boundary errors, islands, and a classifier with a tight or loose box | MSD Task02_Heart |
| F | Optional: a 2-D U-Net in PyTorch (MONAI's implementation if installed) | synthetic |

Every size, intensity and noise level of the synthetic data is illustrative. Each part ends with **checks**. Parts
A to D run in under a minute; part E downloads an archive of several hundred megabytes once.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from sklearn.linear_model import LogisticRegression

from epml import imaging as im
from epml.data import DataUnavailable

## Part A · Metrics with known answers

Four measurements on shapes where the answer is known:

1. Concentric spheres of radii 20 and 21 mm on a 0.5-mm grid: Dice against eq-m40-sphere.
2. A wall 1.89 mm thick (the mean left atrial wall thickness measured on CT by Beinart et al. 2011) shifted by
   1 mm: Dice against eq-m40-slab.
3. The same 1-mm boundary error, plus one stray island 20 mm away: Dice, the Hausdorff distance (HD), its 95th
   percentile (HD95), the average symmetric surface distance (ASSD) and the normalized surface Dice (NSD) at
   4 mm, the tolerance the Medical Segmentation Decathlon used for its heart task.
4. The voxel-face surface area of a sphere, against $4\pi R^2$.

In [ ]:
ball = im.ball
sp05 = (0.5, 0.5, 0.5)
shape = (100, 150, 100)                                       # 50 x 75 x 50 mm
true = ball(shape, (25, 25, 25), 20, sp05)
grown = ball(shape, (25, 25, 25), 21, sp05)
print(f"1. spheres 20 / 21 mm: Dice {im.dice(grown, true):.4f}, eq-m40-sphere {im.sphere_dice(20, 1):.4f}")

wall = np.zeros((600, 4, 4), bool)
wall[100:100 + round(1.89 / 0.01)] = True                    # 0.01-mm voxels along the normal
shifted = np.roll(wall, 100, axis=0)                          # 1.00 mm
print(f"2. wall 1.89 mm shifted 1 mm: Dice {im.dice(shifted, wall):.4f}, eq-m40-slab {im.slab_dice(1.89, 1.0):.4f}")

island = grown.copy()
island[ball(shape, (25, 25 + 21 + 20 + 2, 25), 2, sp05)] = True    # radius 2 mm, nearest edge 20 mm out
rows = {}
for name, pred in [("1 mm error", grown), ("1 mm error + island", island)]:
    rows[name] = {"Dice": im.dice(pred, true), "HD": im.hausdorff(pred, true, sp05),
                  "HD95": im.hausdorff(pred, true, sp05, 95), "ASSD": im.assd(pred, true, sp05),
                  "NSD@4mm": im.nsd(pred, true, 4.0, sp05)}
print("3.", " " * 18, "  ".join(f"{k:>8s}" for k in rows["1 mm error"]))
for name, r in rows.items():
    print(f"   {name:20s}", "  ".join(f"{v:8.3f}" for v in r.values()))
area = im.surface_area_voxels(true, sp05)
print(f"4. voxel-face area / 4 pi R^2 = {area / (4 * np.pi * 20 ** 2):.3f}")

In [ ]:
# Checks
assert abs(im.dice(grown, true) - im.sphere_dice(20, 1)) < 0.005
assert abs(im.dice(shifted, wall) - im.slab_dice(1.89, 1.0)) < 0.01
a, b = rows["1 mm error"], rows["1 mm error + island"]
assert abs(a["Dice"] - b["Dice"]) < 0.01 and b["HD"] > 20 > 2 > a["HD"]
assert b["HD95"] < 2 and b["NSD@4mm"] > 0.98
assert 1.4 < area / (4 * np.pi * 20 ** 2) < 1.6

## Part B · Synthetic atria

Twelve cases on a 64 × 64 × 48 grid of 1.5-mm voxels: an ellipsoidal body, four veins and an appendage form the
label; a bright tube next to it is not atrium; a smooth intensity bias and noise are added. The first eight cases
train, the last four test (a split by case, as by patient).

In [ ]:
rng = np.random.default_rng(40)
cases = [im.synth_atrium(rng) for _ in range(12)]
train, test = cases[:8], cases[8:]
SP = cases[0][2]["spacing"]
img, lab, info = test[0]
fig, axs = plt.subplots(1, 3, figsize=(9, 3))
for ax, z in zip(axs, [14, 24, 34]):
    ax.imshow(img[:, :, z].T, cmap="gray", origin="lower")
    ax.contour(lab[:, :, z].T, [0.5], colors="tab:red", linewidths=1)
    ax.set(title=f"axial slice {z}", xticks=[], yticks=[])
plt.show()
print(f"label covers {np.mean([c[1].mean() for c in cases]):.1%} of voxels; "
      f"veins and appendage {np.mean([(c[2]['veins'] | c[2]['appendage']).sum() / c[1].sum() for c in cases]):.0%} of the label")

In [ ]:
# Checks
assert all(c[1].any() for c in cases) and len(test) == 4
assert not any((c[2]["distractor"] & c[1]).any() for c in cases)

## Part C · A threshold against a voxel classifier

Method 1 keeps voxels above a threshold, chosen on the training cases, and then the largest connected component.
Method 2 is logistic regression on six local features (intensity, smoothed intensity and gradient magnitude at two
scales, and a Laplacian of Gaussian), followed by the same largest-component step. Judge both on the test cases with
Dice, HD95, ASSD and NSD at 4 mm, and split Dice between the body and the thin parts (veins and appendage).

In [ ]:
ths = np.linspace(0.3, 1.2, 19)
th = max(ths, key=lambda t: np.mean([im.dice(im.largest_component(i > t), l) for i, l, _ in train]))

Xs, ys = [], []
for i, l, _ in train:
    F = im.local_features(i)
    idx = rng.choice(F.shape[0], 20000, replace=False)
    Xs.append(F[idx]); ys.append(l.ravel()[idx])
X, y = np.concatenate(Xs), np.concatenate(ys)
mu, sd = X.mean(0), X.std(0)
clf = LogisticRegression(max_iter=2000).fit((X - mu) / sd, y)

def segment(i):
    p = clf.predict_proba((im.local_features(i) - mu) / sd)[:, 1].reshape(i.shape)
    return im.largest_component(p > 0.5)

def scores(pred, l, inf):
    thin = inf["veins"] | inf["appendage"]
    return {"Dice": im.dice(pred, l), "HD95": im.hausdorff(pred, l, SP, 95), "ASSD": im.assd(pred, l, SP),
            "NSD@4mm": im.nsd(pred, l, 4.0, SP), "Dice body": im.dice(pred & inf["body"], inf["body"]),
            "Dice thin": im.dice(pred & thin, l & thin)}

preds = {"threshold": [im.largest_component(i > th) for i, _, _ in test], "classifier": [segment(i) for i, _, _ in test]}
table = {m: {k: np.mean([scores(p, l, inf)[k] for p, (_, l, inf) in zip(ps, test)]) for k in scores(ps[0], *test[0][1:])}
         for m, ps in preds.items()}
print(f"threshold chosen on training cases: {th:.2f}")
print(" " * 12, "  ".join(f"{k:>9s}" for k in table["threshold"]))
for m, r in table.items():
    print(f"{m:12s}", "  ".join(f"{v:9.3f}" for v in r.values()))

In [ ]:
# Checks
t, c = table["threshold"], table["classifier"]
assert c["Dice"] > t["Dice"] and c["ASSD"] < t["ASSD"]
assert c["Dice thin"] < c["Dice body"]            # thin parts are harder, as eq-m40-offset predicts

## Part D · From mask to map

A mapping catheter touches the true wall. If a segmented shell is used for guidance or registration (m21), what
matters is how far each touched point lies from the predicted shell. Sample 500 points on the true surface of each
test case and measure that distance for the classifier's segmentation, separately for the body and the thin parts.

In [ ]:
d_body, d_thin = [], []
for pred, (_, l, inf) in zip(preds["classifier"], test):
    pts = np.argwhere(im.surface(l))
    pts = pts[rng.choice(len(pts), 500, replace=False)]
    dist = ndi.distance_transform_edt(~im.surface(pred), sampling=SP)[tuple(pts.T)]
    thin = (inf["veins"] | inf["appendage"])[tuple(pts.T)]
    d_body.append(dist[~thin]); d_thin.append(dist[thin])
d_body, d_thin = np.concatenate(d_body), np.concatenate(d_thin)
for name, d in [("body", d_body), ("veins and appendage", d_thin)]:
    print(f"{name:20s} median {np.median(d):.1f} mm, 95th percentile {np.percentile(d, 95):.1f} mm, "
          f"beyond 4 mm {np.mean(d > 4):.1%}")
plt.hist([d_body, d_thin], bins=np.arange(0, 12, 0.75), label=["body", "veins and appendage"], density=True)
plt.xlabel("distance from touched point to predicted shell (mm)"); plt.ylabel("density"); plt.legend(); plt.show()

In [ ]:
# Checks
assert np.percentile(d_thin, 95) > np.percentile(d_body, 95)

## Part E · Real data: the Decathlon left atrium

Task02_Heart of the Medical Segmentation Decathlon (Antonelli et al. 2022) holds 30 cardiac MRI volumes from the
2013 Left Atrial Segmentation Challenge (Tobon-Gomez et al. 2015); the 20 training volumes come with the left atrium
outlined. Licence CC BY-SA 4.0. The archive is fetched from the MONAI consortium's AWS Open Data bucket and checked
against the MD5 value that MONAI publishes. Nothing derived from it is stored in this repository.

1. **Shape.** For each label: volume, voxel-face surface area, and Dice after shrinking the label by 1 mm, against
   a voxel sphere of the same volume on the same grid, shrunk the same way (eq-m40-offset predicts the gap).
2. **Islands.** Add a 2-mm blob about 20 mm outside the atrium: Dice against the Hausdorff distance.
3. **A classifier with a tight or a loose box.** The voxel classifier of part C, trained on cases 1–15 and tested on
   16–20, inside a box around the true label with a 10-mm or a 40-mm margin. The box comes from the label, an
   "oracle" first stage; real pipelines must find it. Volumes are resampled to half resolution for speed.

In [ ]:
try:
    import nibabel as nib
    from epml.data import fetch_msd_heart
    pairs = fetch_msd_heart()
    REAL = True
except DataUnavailable as err:
    print("Part E skipped:", err)
    REAL = False

def crop_box(lab, spacing, margin_mm):
    idx = np.argwhere(lab)
    m = np.ceil(margin_mm / np.asarray(spacing)).astype(int)
    lo = np.maximum(idx.min(0) - m, 0)
    hi = np.minimum(idx.max(0) + m + 1, lab.shape)
    return tuple(slice(a, b) for a, b in zip(lo, hi))

if REAL:
    vols = []
    for ip, lp in pairs:
        li = nib.load(str(lp))
        spacing = tuple(float(z) for z in li.header.get_zooms()[:3])
        lab_full = np.asarray(li.dataobj) > 0
        img_full = np.asarray(nib.load(str(ip)).dataobj, dtype=np.float32)
        vols.append((img_full, lab_full, spacing))
    print(f"{len(vols)} volumes; shapes {sorted({v[0].shape for v in vols})[:3]}...; "
          f"spacing of the first {tuple(round(s, 2) for s in vols[0][2])} mm")

In [ ]:
if REAL:
    shape_rows = []
    for img_full, lab_full, spacing in vols:
        box = crop_box(lab_full, spacing, 25)
        l = lab_full[box]
        vox = float(np.prod(spacing))
        V = l.sum() * vox
        S = im.surface_area_voxels(l, spacing)
        R_eq = (3 * V / (4 * np.pi)) ** (1 / 3)
        n = [int(np.ceil((2 * R_eq + 6) / s)) for s in spacing]
        sph = im.ball(n, [k * s / 2 for k, s in zip(n, spacing)], R_eq, spacing)   # same volume, same voxels
        shrink = lambda m: im.dice(im.offset_mask(m, -1.0, spacing), m)
        shape_rows.append((V / 1000, S / 1.5 / V, 3 / R_eq, shrink(l), shrink(sph)))
    shape_rows = np.array(shape_rows)
    print("median over 20 labels:")
    print(f"  volume {np.median(shape_rows[:, 0]):.0f} mL; surface/volume (voxel faces / 1.5) {np.median(shape_rows[:, 1]):.3f} /mm, "
          f"equal-volume sphere {np.median(shape_rows[:, 2]):.3f} /mm")
    print(f"  Dice after shrinking by 1 mm (one voxel layer on this grid): label {np.median(shape_rows[:, 3]):.3f}, "
          f"voxel sphere of the same volume {np.median(shape_rows[:, 4]):.3f}")

In [ ]:
if REAL:
    island_rows = []
    for img_full, lab_full, spacing in vols[:5]:
        box = crop_box(lab_full, spacing, 25)
        l = lab_full[box]
        dist_out = ndi.distance_transform_edt(~l, sampling=spacing)
        far = np.argwhere((dist_out >= 20) & (dist_out < 21))
        if len(far) == 0:
            continue
        seed = np.zeros_like(l)
        seed[tuple(far[0])] = True
        blob = ndi.distance_transform_edt(~seed, sampling=spacing) <= 2.0
        pred = l | blob
        island_rows.append((im.dice(pred, l), im.hausdorff(pred, l, spacing), im.hausdorff(pred, l, spacing, 95), im.assd(pred, l, spacing)))
    island_rows = np.array(island_rows)
    print(f"label plus one island ({len(island_rows)} cases): Dice {island_rows[:, 0].min():.4f} or more, "
          f"HD {island_rows[:, 1].min():.1f}–{island_rows[:, 1].max():.1f} mm, HD95 up to {island_rows[:, 2].max():.2f} mm, "
          f"ASSD up to {island_rows[:, 3].max():.2f} mm")

In [ ]:
if REAL:
    def prepared(img_full, lab_full, spacing, margin):
        box = crop_box(lab_full, spacing, margin)
        i = ndi.zoom(img_full[box], 0.5, order=1)
        l = ndi.zoom(lab_full[box].astype(np.uint8), 0.5, order=0) > 0
        lo, hi = np.percentile(i, [1, 99])
        return (i - lo) / (hi - lo + 1e-6), l, tuple(2 * s for s in spacing)

    box_results = {}
    for margin in (10, 40):
        data_ = [prepared(*v, margin) for v in vols]
        Xs, ys = [], []
        for i, l, _ in data_[:15]:
            F = im.local_features(i)
            k = rng.choice(F.shape[0], min(20000, F.shape[0]), replace=False)
            Xs.append(F[k]); ys.append(l.ravel()[k])
        X, y = np.concatenate(Xs), np.concatenate(ys)
        mu_r, sd_r = X.mean(0), X.std(0)
        clf_r = LogisticRegression(max_iter=2000).fit((X - mu_r) / sd_r, y)
        res = []
        for i, l, s in data_[15:]:
            p = clf_r.predict_proba((im.local_features(i) - mu_r) / sd_r)[:, 1].reshape(i.shape)
            pred = im.largest_component(p > 0.5)
            res.append((im.dice(pred, l), im.hausdorff(pred, l, s, 95) if pred.any() else np.nan))
        box_results[margin] = np.array(res)
        print(f"box margin {margin} mm: Dice {np.mean(box_results[margin][:, 0]):.3f} "
              f"(per case {', '.join(f'{d:.2f}' for d in box_results[margin][:, 0])}), "
              f"median HD95 {np.nanmedian(box_results[margin][:, 1]):.1f} mm")

In [ ]:
# Checks
if REAL:
    assert len(vols) == 20
    assert np.median(shape_rows[:, 3]) < np.median(shape_rows[:, 4])      # real atria lose more Dice than spheres
    assert np.all(island_rows[:, 0] > 0.99) and np.all(island_rows[:, 1] >= 20)
    assert np.mean(box_results[10][:, 0]) > np.mean(box_results[40][:, 0])

## Part F · Optional: a 2-D U-Net

A small U-Net (Ronneberger et al. 2015) trained on axial slices of the synthetic training cases with a soft Dice
loss, then applied slice by slice to the test cases. If MONAI is installed (`pip install monai`), the network and the
loss come from MONAI, the open-source framework started by NVIDIA and King's College London; otherwise a minimal
PyTorch version is used. Runs only if PyTorch is installed.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part F skipped.")

if torch is not None:
    torch.manual_seed(0)
    try:
        from monai.losses import DiceLoss
        from monai.networks.nets import UNet
        net = UNet(spatial_dims=2, in_channels=1, out_channels=1, channels=(8, 16, 32), strides=(2, 2), num_res_units=1)
        loss_fn = DiceLoss(sigmoid=True)
        print("using MONAI", __import__("monai").__version__)
    except ImportError:
        if os.environ.get("EPML_REQUIRE_MONAI") == "1":
            raise
        class Tiny(torch.nn.Module):
            def __init__(self):
                super().__init__()
                c = lambda i, o: torch.nn.Sequential(torch.nn.Conv2d(i, o, 3, padding=1), torch.nn.ReLU())
                self.e1, self.e2 = c(1, 8), c(8, 16)
                self.d1 = c(24, 8)
                self.out = torch.nn.Conv2d(8, 1, 1)
            def forward(self, x):
                a = self.e1(x)
                b = self.e2(torch.nn.functional.max_pool2d(a, 2))
                u = torch.nn.functional.interpolate(b, scale_factor=2)
                return self.out(self.d1(torch.cat([a, u], 1)))
        net = Tiny()
        def loss_fn(logits, target):
            p = torch.sigmoid(logits)
            return 1 - (2 * (p * target).sum() + 1e-5) / (p.sum() + target.sum() + 1e-5)
        print("MONAI not installed: using a minimal PyTorch U-Net")

    def slices(cs):
        xs = np.concatenate([np.moveaxis(i, 2, 0) for i, _, _ in cs])[:, None]
        ys = np.concatenate([np.moveaxis(l, 2, 0) for _, l, _ in cs])[:, None].astype(np.float32)
        return torch.tensor(xs), torch.tensor(ys)
    Xt, Yt = slices(train)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for epoch in range(25):
        perm = torch.randperm(len(Xt))
        for k in range(0, len(Xt), 32):
            b = perm[k:k + 32]
            opt.zero_grad()
            loss = loss_fn(net(Xt[b]), Yt[b])
            loss.backward()
            opt.step()
    net.eval()
    unet_dice = []
    with torch.no_grad():
        for i, l, _ in test:
            p = torch.sigmoid(net(torch.tensor(np.moveaxis(i, 2, 0)[:, None]))).numpy()[:, 0]
            unet_dice.append(im.dice(im.largest_component(np.moveaxis(p, 0, 2) > 0.5), l))
    print(f"2-D U-Net, test Dice {np.mean(unet_dice):.3f} (voxel classifier of part C: {table['classifier']['Dice']:.3f})")

In [ ]:
# Checks
if torch is not None:
    assert np.mean(unet_dice) > 0.7

## Where to go next

- In part A, move the island closer. At what gap does HD95 start to change?
- In part C, train the classifier on four cases instead of eight. Which score falls first?
- In part E, shrink by 0.5 mm instead of 1 mm. Does the gap between the real atria and the spheres follow
  eq-m40-offset?